# IoT Asthma Risk Predictor — Upgraded ML Pipeline
### Task 1: Ensemble + Calibrated Probabilities + SHAP Feature Attribution + Isolation Forest Anomaly Gate

**Architecture:**
- **Ensemble:** Random Forest + HistGradientBoosting (sklearn's native XGBoost-equivalent) + Logistic Regression
- **Calibration:** `CalibratedClassifierCV` (isotonic) for reliable risk probabilities
- **Explainability:** Per-sample SHAP-style feature attribution via RF leaf contributions
- **Anomaly Gate:** `IsolationForest` rejects bad/outlier sensor readings before prediction
- **Output:** Single `.pkl` bundle with all artefacts

> **Note on XGBoost:** `xgboost` is an optional external library.  
> `HistGradientBoostingClassifier` (sklearn built-in) uses the same histogram-based  
> gradient boosting algorithm and achieves identical or better accuracy.  
> If you have `xgboost` installed in your environment, Cell 2 shows how to swap it in.

In [1]:
# ── Imports & environment check ───────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle
from collections import Counter

# sklearn — guaranteed available
import sklearn
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,   # native XGBoost-equivalent, no install needed
    VotingClassifier,
    IsolationForest,
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.inspection import permutation_importance

# SMOTE for class balancing
from imblearn.over_sampling import SMOTE

import matplotlib
matplotlib.use("Agg")          # headless — safe for all environments
import matplotlib.pyplot as plt

print(f"scikit-learn : {sklearn.__version__}")
print(f"numpy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")

# Optional: XGBoost
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    import xgboost as xgb
    print(f"xgboost      : {xgb.__version__}  ✓ (will be used in ensemble)")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("xgboost      : not installed → using HistGradientBoostingClassifier instead")

# Optional: SHAP
try:
    import shap
    SHAP_AVAILABLE = True
    print(f"shap         : {shap.__version__}  ✓")
except ImportError:
    SHAP_AVAILABLE = False
    print("shap         : not installed → using RF-importance proxy for feature attribution")

print("\n✅ Environment check complete")


scikit-learn : 1.7.2
numpy        : 1.26.4
pandas       : 2.3.0
xgboost      : 3.0.5  ✓ (will be used in ensemble)
shap         : 0.49.1  ✓

✅ Environment check complete


## Step 1 — Load Data & Balance Classes with SMOTE

In [2]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv("final_asthma_dataset.csv")

FEATURES = ["air_quality", "temperature", "humidity", "spo2", "heart_rate"]
TARGET   = "asthma_risk"

X = df[FEATURES]
y = df[TARGET]

print("Dataset shape:", df.shape)
print("Class distribution before SMOTE:", Counter(y))
print("\nFeature statistics:")
print(X.describe().round(2))

# ── Balance with SMOTE ────────────────────────────────────────────────────────
sm = SMOTE(random_state=42)
X_bal, y_bal = sm.fit_resample(X, y)
print("\nClass distribution after SMOTE :", Counter(y_bal))

# ── Train / test split (stratified) ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")


Dataset shape: (2392, 6)
Class distribution before SMOTE: Counter({0: 2268, 1: 124})

Feature statistics:
       air_quality  temperature  humidity     spo2  heart_rate
count      2392.00      2392.00   2392.00  2392.00     2392.00
mean          5.01        36.76     59.77    97.46       79.09
std           2.94         0.43     11.54     1.44       11.54
min           0.00        36.00     40.03    95.00       60.00
25%           2.43        36.39     49.71    96.22       69.00
50%           5.04        36.76     59.60    97.43       79.00
75%           7.63        37.15     69.69    98.67       89.00
max          10.00        37.50     79.99   100.00       99.00

Class distribution after SMOTE : Counter({0: 2268, 1: 2268})
Train: 3628 samples  |  Test: 908 samples


## Step 2 — Build Ensemble & Calibrate Probabilities

**Why `CalibratedClassifierCV` around a `VotingClassifier`?**  
`VotingClassifier(voting='soft')` averages raw probabilities from each sub-model,  
which are often poorly calibrated. Wrapping with `CalibratedClassifierCV(method='isotonic')`  
applies isotonic regression to map those scores to true posterior probabilities — critical  
for a medical risk application where "70% risk" should mean something real.

In [3]:
# ── Base models ───────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)

# Use XGBoost if available, otherwise HistGradientBoosting (same algorithm)
if XGBOOST_AVAILABLE:
    boost = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        eval_metric="logloss", random_state=42, n_jobs=-1,
        verbosity=0,
    )
    boost_name = "xgb"
else:
    boost = HistGradientBoostingClassifier(
        max_iter=200, max_depth=6, learning_rate=0.05,
        random_state=42,
    )
    boost_name = "hgb"

# Logistic Regression needs scaling → wrap in Pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
])

# ── Soft-voting ensemble ──────────────────────────────────────────────────────
ensemble = VotingClassifier(
    estimators=[("rf", rf), (boost_name, boost), ("lr", lr_pipeline)],
    voting="soft",          # average predicted probabilities
)

# ── Calibrate for reliable probabilities ─────────────────────────────────────
# cv=5 folds: each fold trains the ensemble, fits an isotonic calibrator.
# Result: well-calibrated P(high risk | features).
calibrated_model = CalibratedClassifierCV(ensemble, method="isotonic", cv=5)

print(f"Ensemble: RandomForest + {boost_name.upper()} + LogisticRegression")
print("Calibration: isotonic, cv=5")
print("Training... (expect 30–90 seconds)")

calibrated_model.fit(X_train, y_train)
print("✅ Training complete")

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred  = calibrated_model.predict(X_test)
y_proba = calibrated_model.predict_proba(X_test)[:, 1]

print("\n── Classification Report ─────────────────────────────────")
print(classification_report(y_test, y_pred, target_names=["Low Risk", "High Risk"]))
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_proba):.4f}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Low Risk", "High Risk"],
    colorbar=False, ax=ax
)
ax.set_title("Confusion Matrix — Calibrated Ensemble")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.close()
print("Confusion matrix saved → confusion_matrix.png")


Ensemble: RandomForest + XGB + LogisticRegression
Calibration: isotonic, cv=5
Training... (expect 30–90 seconds)
✅ Training complete

── Classification Report ─────────────────────────────────
              precision    recall  f1-score   support

    Low Risk       0.87      0.83      0.85       454
   High Risk       0.84      0.88      0.86       454

    accuracy                           0.85       908
   macro avg       0.85      0.85      0.85       908
weighted avg       0.85      0.85      0.85       908

ROC-AUC Score : 0.9189
Confusion matrix saved → confusion_matrix.png


## Step 3 — Isolation Forest Anomaly Gate

**Why?** NodeMCU + MQ-135 sensors occasionally produce nonsensical readings  
(wire glitch, ADC saturation, sensor warm-up transient). Feeding garbage into  
the ML model gives confident but meaningless predictions.  

The Isolation Forest is trained on clean sensor data. At inference time,  
any reading it scores as an outlier is **rejected before** reaching the classifier.

In [4]:
# ── Train Isolation Forest on training features ───────────────────────────────
# contamination=0.03 → expects ~3% of live readings to be outliers.
# Tune this to your hardware's observed noise rate.
anomaly_gate = IsolationForest(
    n_estimators=200,
    contamination=0.03,      # ← adjust based on your sensor reliability
    max_samples="auto",
    random_state=42,
    n_jobs=-1,
)
anomaly_gate.fit(X_train)

# ── Validate: how many test samples are flagged as anomalies? ─────────────────
test_flags = anomaly_gate.predict(X_test)   # 1 = normal, -1 = anomaly
n_flagged  = (test_flags == -1).sum()
print(f"Anomaly gate — flagged {n_flagged}/{len(X_test)} test samples "
      f"({100 * n_flagged / len(X_test):.1f}%) as outliers")

# ── Synthetic sensor fault examples ──────────────────────────────────────────
sensor_faults = pd.DataFrame({
    "air_quality":  [999, -5,   350],     # ADC saturated | negative | normal
    "temperature":  [85,   22,   30],     # impossible     | ok       | ok
    "humidity":     [101,  55,   60],     # >100%          | ok       | ok
    "spo2":         [20,   98,   97],     # impossibly low | ok       | ok
    "heart_rate":   [240,  72,   80],     # impossibly high| ok       | ok
})
fault_flags = anomaly_gate.predict(sensor_faults)
labels = {1: "✅ NORMAL", -1: "🚨 ANOMALY"}
print("\nSynthetic fault detection:")
for i, (flag, row) in enumerate(zip(fault_flags, sensor_faults.itertuples(index=False))):
    print(f"  Sample {i}: {labels[flag]}  {dict(zip(FEATURES, row))}")


Anomaly gate — flagged 32/908 test samples (3.5%) as outliers

Synthetic fault detection:
  Sample 0: 🚨 ANOMALY  {'air_quality': 999, 'temperature': 85, 'humidity': 101, 'spo2': 20, 'heart_rate': 240}
  Sample 1: ✅ NORMAL  {'air_quality': -5, 'temperature': 22, 'humidity': 55, 'spo2': 98, 'heart_rate': 72}
  Sample 2: ✅ NORMAL  {'air_quality': 350, 'temperature': 30, 'humidity': 60, 'spo2': 97, 'heart_rate': 80}


## Step 4 — Per-Sample Feature Attribution

**If `shap` is installed:** uses `TreeExplainer` on the RF component (exact SHAP values).  
**If not:** uses a mathematically principled proxy:  
`contribution_i = feature_importance_i × |x_i − μ_train_i|`  
— measures how far each feature deviates from the training mean, weighted by  
the RF's learned importance for that feature.  
Both produce a signed value: **positive = pushes toward HIGH RISK**.

In [5]:
# ── Train a standalone RF for feature attribution ─────────────────────────────
# We use a separate RF (not the calibrated ensemble) because SHAP / feature
# attribution needs direct access to the tree structure, which is hidden inside
# CalibratedClassifierCV's internal clones.
rf_explainer_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=3,
    random_state=42, n_jobs=-1,
)
rf_explainer_model.fit(X_train, y_train)
print("Explainer RF trained")
print(f"Explainer RF accuracy on test set: "
      f"{rf_explainer_model.score(X_test, y_test):.3f}")

# ── Build the right explainer ─────────────────────────────────────────────────
if SHAP_AVAILABLE:
    print("Using SHAP TreeExplainer (exact values)")
    _shap_explainer = shap.TreeExplainer(rf_explainer_model)

    def _get_attribution(sample_array):
        """
        Returns dict: feature_name → signed SHAP value (class-1 / high-risk)
        sample_array: np.ndarray shape (1, 5)
        """
        sv = _shap_explainer.shap_values(sample_array)
        if isinstance(sv, list):
            vals = np.array(sv[1]).flatten()
        else:
            sv = np.array(sv)
            if sv.ndim == 3:
                vals = sv[0, :, 1]
            elif sv.ndim == 2:
                vals = sv[0]
            else:
                vals = sv.flatten()
        return dict(zip(FEATURES, vals.tolist()))
else:
    print("Using RF-importance proxy (SHAP not installed)")
    _rf_importances = rf_explainer_model.feature_importances_
    _train_mean     = np.array(X_train).mean(axis=0)

    def _get_attribution(sample_array):
        """
        Proxy: signed_contribution_i = importance_i × (x_i − mean_i)
        Positive = feature value above normal average → pushes toward high risk
        Negative = feature value below normal average → pushes toward low risk
        """
        deviation = sample_array[0] - _train_mean
        contribs  = _rf_importances * deviation
        return dict(zip(FEATURES, contribs.tolist()))


def get_top_shap_feature(sample_array):
    """
    Public API used by the inference pipeline.
    
    Parameters
    ----------
    sample_array : np.ndarray, shape (1, 5)
        One sensor reading: [air_quality, temperature, humidity, spo2, heart_rate]
    
    Returns
    -------
    top_feature : str   — name of the most influential feature
    shap_value  : float — signed attribution (positive = toward HIGH RISK)
    all_values  : dict  — {feature: attribution} for all 5 features
    """
    attrs    = _get_attribution(sample_array)
    top_feat = max(attrs, key=lambda k: abs(attrs[k]))
    return top_feat, attrs[top_feat], attrs


# ── Quick sanity check ────────────────────────────────────────────────────────
sample = np.array(X_test.iloc[[0]])
feat, val, all_attrs = get_top_shap_feature(sample)
print(f"\nSample 0 attributions: {all_attrs}")
print(f"Top feature: '{feat}'  (value: {val:+.4f})")


Explainer RF trained
Explainer RF accuracy on test set: 0.760
Using SHAP TreeExplainer (exact values)

Sample 0 attributions: {'air_quality': -0.012175262410652133, 'temperature': -0.22751060859842195, 'humidity': -0.05747245709500644, 'spo2': -0.1140456055611116, 'heart_rate': -0.005207770484961243}
Top feature: 'temperature'  (value: -0.2275)


## Step 5 — Full Inference Pipeline

One function that combines the anomaly gate + calibrated model + feature attribution.  
This is what your Flask/FastAPI server (Task 2) will call.

In [6]:
def predict_asthma_risk(sensor_reading: dict) -> dict:
    """
    End-to-end inference with anomaly gating and feature attribution.

    Parameters
    ----------
    sensor_reading : dict with keys:
        air_quality, temperature, humidity, spo2, heart_rate

    Returns
    -------
    dict with:
        status         : "ok" | "anomaly_rejected"
        prediction     : 0 (low) | 1 (high) | None if rejected
        risk_label     : "LOW RISK" | "HIGH RISK" | "SENSOR ERROR"
        probability    : float P(high risk) | None if rejected
        top_feature    : str | None
        shap_value     : float | None
        all_attributions: dict | None
        anomaly_score  : float (lower = more anomalous)
    """
    arr = np.array([[
        sensor_reading["air_quality"],
        sensor_reading["temperature"],
        sensor_reading["humidity"],
        sensor_reading["spo2"],
        sensor_reading["heart_rate"],
    ]])

    # ── Stage 1: Anomaly gate ─────────────────────────────────────────────────
    iso_flag    = anomaly_gate.predict(arr)[0]       # 1=normal, -1=anomaly
    iso_score   = float(anomaly_gate.score_samples(arr)[0])

    if iso_flag == -1:
        return {
            "status":          "anomaly_rejected",
            "prediction":      None,
            "risk_label":      "SENSOR ERROR — reading rejected",
            "probability":     None,
            "top_feature":     None,
            "shap_value":      None,
            "all_attributions": None,
            "anomaly_score":   iso_score,
        }

    # ── Stage 2: Calibrated ensemble prediction ───────────────────────────────
    pred  = int(calibrated_model.predict(arr)[0])
    proba = float(calibrated_model.predict_proba(arr)[0][1])

    # ── Stage 3: Feature attribution ─────────────────────────────────────────
    top_feat, shap_val, all_attrs = get_top_shap_feature(arr)

    return {
        "status":           "ok",
        "prediction":       pred,
        "risk_label":       "HIGH RISK" if pred == 1 else "LOW RISK",
        "probability":      round(proba, 4),
        "top_feature":      top_feat,
        "shap_value":       round(shap_val, 4),
        "all_attributions": {k: round(v, 4) for k, v in all_attrs.items()},
        "anomaly_score":    iso_score,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
print("── Inference pipeline demo ────────────────────────────────────────────")
print(f"{'Sample':<8} {'Status':<20} {'Label':<12} {'Prob':<8} {'Top Feature':<15} {'Attribution'}")
print("─" * 80)

test_cases = [
    # Normal-range samples from X_test
    *[dict(zip(FEATURES, X_test.iloc[i].tolist())) for i in range(5)],
    # Synthetic fault
    {"air_quality": 999, "temperature": 85, "humidity": 101, "spo2": 20, "heart_rate": 240},
    {"air_quality": -10, "temperature": 22, "humidity": 55,  "spo2": 98, "heart_rate": 72},
]

for i, reading in enumerate(test_cases):
    r = predict_asthma_risk(reading)
    if r["status"] == "ok":
        print(f"{i:<8} {'ok':<20} {r['risk_label']:<12} {r['probability']:<8.3f} "
              f"{r['top_feature']:<15} {r['shap_value']:+.4f}")
    else:
        print(f"{i:<8} {'🚨 ANOMALY_REJECTED':<20} {'–':<12} {'–':<8} {'–':<15} –")


── Inference pipeline demo ────────────────────────────────────────────
Sample   Status               Label        Prob     Top Feature     Attribution
────────────────────────────────────────────────────────────────────────────────
0        ok                   LOW RISK     0.000    temperature     -0.2275
1        ok                   LOW RISK     0.174    temperature     -0.0801
2        ok                   LOW RISK     0.000    heart_rate      -0.2490
3        ok                   HIGH RISK    0.916    spo2            +0.0836
4        ok                   HIGH RISK    0.662    temperature     -0.1572
5        🚨 ANOMALY_REJECTED   –            –        –               –
6        ok                   LOW RISK     0.068    temperature     -0.3292


## Step 6 — Save Everything as `.pkl`

The bundle contains **all artefacts needed for deployment** — no re-training required.

In [7]:
# ── Assemble the deployment bundle ────────────────────────────────────────────
bundle = {
    # ── Models ───────────────────────────────────────────────────────────────
    "calibrated_ensemble":  calibrated_model,      # main predictor
    "anomaly_gate":         anomaly_gate,           # Isolation Forest
    "rf_explainer":         rf_explainer_model,     # for feature attribution
    
    # ── Metadata ─────────────────────────────────────────────────────────────
    "feature_names":        FEATURES,
    "target_name":          TARGET,
    "classes":              [0, 1],
    "class_labels":         {0: "LOW RISK", 1: "HIGH RISK"},
    
    # ── Training stats (useful for monitoring drift) ─────────────────────────
    "train_feature_mean":   np.array(X_train).mean(axis=0).tolist(),
    "train_feature_std":    np.array(X_train).std(axis=0).tolist(),
    "train_size":           len(X_train),
    
    # ── SHAP proxy info ───────────────────────────────────────────────────────
    "rf_feature_importances": rf_explainer_model.feature_importances_.tolist(),
    "shap_available":         SHAP_AVAILABLE,
    
    # ── Performance snapshot ─────────────────────────────────────────────────
    "test_roc_auc":         round(roc_auc_score(y_test, y_proba), 4),
    
    # ── Inference function (convenience) ─────────────────────────────────────
    "predict_fn":           predict_asthma_risk,
}

PKL_PATH = "asthma_risk_model.pkl"
with open(PKL_PATH, "wb") as f:
    pickle.dump(bundle, f, protocol=4)

import os
size_kb = os.path.getsize(PKL_PATH) / 1024
print(f"✅  Saved → {PKL_PATH}  ({size_kb:.1f} KB)")
print()
print("Bundle keys:")
for k, v in bundle.items():
    vtype = type(v).__name__
    print(f"  {k:<35} {vtype}")


✅  Saved → asthma_risk_model.pkl  (16780.6 KB)

Bundle keys:
  calibrated_ensemble                 CalibratedClassifierCV
  anomaly_gate                        IsolationForest
  rf_explainer                        RandomForestClassifier
  feature_names                       list
  target_name                         str
  classes                             list
  class_labels                        dict
  train_feature_mean                  list
  train_feature_std                   list
  train_size                          int
  rf_feature_importances              list
  shap_available                      bool
  test_roc_auc                        float
  predict_fn                          function


## Step 7 — Reload & Verify the Bundle

Always verify your pkl round-trips correctly before shipping to production.

In [8]:
# ── Reload ────────────────────────────────────────────────────────────────────
with open("asthma_risk_model.pkl", "rb") as f:
    loaded = pickle.load(f)

print("Loaded keys:", list(loaded.keys()))
print(f"Test ROC-AUC (from bundle): {loaded['test_roc_auc']}")
print(f"Feature names: {loaded['feature_names']}")
print(f"Trained on {loaded['train_size']} samples")

# ── End-to-end round-trip test ────────────────────────────────────────────────
predict_reloaded = loaded["predict_fn"]

sample_high_risk = {
    "air_quality": 380,   # high pollution
    "temperature": 35,    # hot
    "humidity":    85,    # humid
    "spo2":        92,    # low blood oxygen
    "heart_rate":  110,   # elevated HR
}
sample_low_risk = {
    "air_quality": 50,
    "temperature": 25,
    "humidity":    45,
    "spo2":        99,
    "heart_rate":  70,
}

print("\n── Round-trip predictions ─────────────────────────────────────────────")
for name, reading in [("High-risk scenario", sample_high_risk), 
                       ("Low-risk scenario",  sample_low_risk)]:
    result = predict_reloaded(reading)
    print(f"\n{name}:")
    print(f"  Risk label   : {result['risk_label']}")
    print(f"  Probability  : {result['probability']:.1%}")
    print(f"  Top feature  : {result['top_feature']}  ({result['shap_value']:+.4f})")
    print(f"  All features : {result['all_attributions']}")

print("\n✅  Bundle verified — ready for Task 2 (cloud deployment)")


Loaded keys: ['calibrated_ensemble', 'anomaly_gate', 'rf_explainer', 'feature_names', 'target_name', 'classes', 'class_labels', 'train_feature_mean', 'train_feature_std', 'train_size', 'rf_feature_importances', 'shap_available', 'test_roc_auc', 'predict_fn']
Test ROC-AUC (from bundle): 0.9189
Feature names: ['air_quality', 'temperature', 'humidity', 'spo2', 'heart_rate']
Trained on 3628 samples

── Round-trip predictions ─────────────────────────────────────────────

High-risk scenario:
  Risk label   : SENSOR ERROR — reading rejected


TypeError: unsupported format string passed to NoneType.__format__

## (Optional) Global SHAP Summary Plot

Run only if `shap` is installed. Shows overall feature importance across the test set.

In [9]:
if SHAP_AVAILABLE:
    shap_values_all = _shap_explainer.shap_values(np.array(X_test))
    if isinstance(shap_values_all, list):
        sv_class1 = shap_values_all[1]
    else:
        sv_arr = np.array(shap_values_all)
        sv_class1 = sv_arr[:, :, 1] if sv_arr.ndim == 3 else sv_arr

    shap.summary_plot(
        sv_class1,
        np.array(X_test),
        feature_names=FEATURES,
        show=False,
        plot_type="bar",
    )
    plt.tight_layout()
    plt.savefig("shap_summary.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("SHAP summary saved → shap_summary.png")
else:
    # Fallback: RF feature importance bar chart
    importances = rf_explainer_model.feature_importances_
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(FEATURES, importances, color="steelblue")
    ax.bar_label(bars, fmt="%.3f", padding=3)
    ax.set_xlabel("Mean Decrease in Impurity (MDI)")
    ax.set_title("RF Feature Importances\n(SHAP proxy — install `shap` for exact values)")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=120)
    plt.close()
    print("Feature importance chart saved → feature_importance.png")
    print("\nFeature importances:")
    for feat, imp in sorted(zip(FEATURES, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 50)
        print(f"  {feat:<15} {imp:.4f}  {bar}")


SHAP summary saved → shap_summary.png
